# Análisis del Experimento: Generación de Niveles de Sokoban
Este notebook procesa `exp1_raw_data.csv` y responde a:
* **RQ1**: Rendimiento agregado entre metaheurísticas.
* **RQ2**: Robustez (varianza intra-algoritmo entre funciones objetivo).
* **Diversidad y Tiempos**: Capacidad de exploración vs Costo.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid')

# Carga de datos
df = pd.read_csv('exp1_raw_data.csv')
df = df.rename(columns={'Fitness_Bruto': 'Fitness', 'BT_id': 'Shell_ID', 'Semilla_ID': 'Rep', 'Tiempo_Segundos': 'Time_s'})
df.head()

## RQ1: Rendimiento Agregado
Normalizamos (Min-Max) el fitness por (FO, Shell) para comparar el rendimiento de los algoritmos globalmente.

In [ ]:
df_norm = df.copy()
df_norm['Fitness_norm'] = np.nan

for (fo, sid), grp in df.groupby(['FO', 'Shell_ID']):
    mn, mx = grp['Fitness'].min(), grp['Fitness'].max()
    if mx > mn:
        df_norm.loc[grp.index, 'Fitness_norm'] = (grp['Fitness'] - mn) / (mx - mn)
    else:
        df_norm.loc[grp.index, 'Fitness_norm'] = 0.5

plt.figure(figsize=(8, 5))
sns.boxplot(data=df_norm, x='Algoritmo', y='Fitness_norm', palette='Set2')
plt.title('RQ1: Distribución Fitness Normalizado Global')
plt.ylabel('Fitness Normalizado [0-1]')
plt.show()

agg = df_norm.groupby('Algoritmo')['Fitness_norm'].agg(['mean', 'std', 'median']).sort_values('mean', ascending=False)
display(agg)

### Test Estadístico (Kruskal-Wallis)
Verificamos diferencias significativas.

In [ ]:
groups = [df_norm[df_norm['Algoritmo'] == alg]['Fitness_norm'].dropna() for alg in df_norm['Algoritmo'].unique()]
stat, p = stats.kruskal(*groups)
print(f'Kruskal-Wallis: H={stat:.4f}, p-value={p:.2e}')

## RQ2: Robustez de la Metaheurística
Medimos el Coeficiente de Variación (CV) entre Funciones Objetivo. Menor CV implica mayor robustez ante cambios en la FO.

In [ ]:
def coef_variation(series):
    m = series.mean()
    return (series.std() / m) if m != 0 else np.nan

mean_per_fo = df_norm.groupby(['Algoritmo', 'FO'])['Fitness_norm'].mean().reset_index()
cv_table = mean_per_fo.groupby('Algoritmo')['Fitness_norm'].apply(coef_variation).sort_values().to_frame('CV (menor = más robusto)')
display(cv_table)

cv_table.plot(kind='bar', color=['#4CAF50', '#FF9800', '#F44336'], edgecolor='black', legend=False)
plt.title('RQ2: Coeficiente de Variación (CV)')
plt.xticks(rotation=0)
plt.show()

plt.figure(figsize=(10, 5))
sns.boxplot(data=df_norm, x='Algoritmo', y='Fitness_norm', hue='FO', palette='Set3')
plt.title('Rendimiento por FO')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

## Diversidad y Costo Computacional
Cantidad de niveles únicos generados (por hash) y tiempo de ejecución.

In [ ]:
diversity = df.groupby(['Algoritmo', 'FO', 'Shell_ID'])['Board_Hash'].nunique().reset_index()
mean_div = diversity.groupby('Algoritmo')['Board_Hash'].mean().sort_values(ascending=False).to_frame('Tableros Únicos Promedio')
mean_time = df.groupby('Algoritmo')['Time_s'].mean().sort_values().to_frame('Tiempo Promedio (s)')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
mean_div.plot(kind='bar', ax=axes[0], color='#2196F3', edgecolor='black', legend=False)
axes[0].set_title('Diversidad Promedio')
mean_time.plot(kind='bar', ax=axes[1], color='#F44336', edgecolor='black', legend=False)
axes[1].set_title('Costo Computacional')
plt.show()

display(mean_div)
display(mean_time)